# Ajuste fino: ResNet-50 original vs ResNet-50 nuevo (320 px, CheXpert)

Este notebook compara el modelo que **entrenamos** en el paso 3 (`resnet50-res320-chex-s0`)
con el modelo de la librería del que partió (`resnet50-res512-all`).

Lee los resultados ya calculados (no vuelve a evaluar):

| Archivo | Qué aporta |
|---|---|
| `modelos/resnet50-res320-chex-s0/historial.csv` y `config.json` | Curva de entrenamiento (paso 3) |
| `resultados/paso2/resultados.json` | Evaluación de los dos modelos con el mismo protocolo (paso 2 con `--agregar`) |
| `resultados/paso2b/tecnicas.json` | Técnicas sin entrenar, como referencia (paso 2b) |

Solo las secciones de **Grad-CAM** (7 y 8) usan la GPU y las imágenes reales de CheXpert y NIH.

**Cómo usarlo:** ábrelo en VS Code y pulsa **Run All**. Las figuras se guardan en
`resultados/figuras/` con los números 9 a 14.

In [ ]:
import os, json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

AQUI = os.path.abspath("")                       # carpeta proyecto/
ORIG = "resnet50-res512-all"
NUEVO = "resnet50-res320-chex-s0"
REF = "densenet121-res224-chex"                  # mejor modelo de la librería en CheXpert
CARPETA_MODELO = os.path.join(AQUI, "modelos", NUEVO)
FIGURAS = os.path.join(AQUI, "resultados", "figuras")
os.makedirs(FIGURAS, exist_ok=True)

with open(os.path.join(AQUI, "resultados", "paso2", "resultados.json"), encoding="utf-8") as f:
    R = json.load(f)
with open(os.path.join(CARPETA_MODELO, "config.json"), encoding="utf-8") as f:
    CFG = json.load(f)
HIST = pd.read_csv(os.path.join(CARPETA_MODELO, "historial.csv"))
T = None
_tec = os.path.join(AQUI, "resultados", "paso2b", "tecnicas.json")
if os.path.exists(_tec):
    with open(_tec, encoding="utf-8") as f:
        T = json.load(f)

if NUEVO not in R["modelos"]:
    raise SystemExit("El modelo nuevo todavía no está en resultados.json. Ejecuten antes:\n"
                     "python proyecto\\paso2_evaluar_base.py --nih ... --chex ... "
                     "--modelos proyecto\\modelos\\resnet50-res320-chex-s0 --agregar")

PATOLOGIAS = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Effusion", "Pneumonia", "Pneumothorax"]
ES = {"Atelectasis": "Atelectasia", "Cardiomegaly": "Cardiomegalia", "Consolidation": "Consolidación",
      "Edema": "Edema", "Effusion": "Derrame", "Pneumonia": "Neumonía", "Pneumothorax": "Neumotórax"}
CONDICIONES = {"nih_original": "NIH original", "nih_320": "NIH 320 px", "chex_valid": "CheXpert valid"}
NOMBRE = {ORIG: "ResNet-50 original", NUEVO: "ResNet-50 nuevo", REF: "DenseNet CheXpert (librería)"}
COLOR = {ORIG: "#2a78d6", NUEVO: "#eb6834", "ref": "#a3a29b", "tec": "#8fb8e8"}

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e4e3df", "grid.linewidth": 0.8,
                     "axes.edgecolor": "#a3a29b", "axes.labelcolor": "#52514e", "xtick.color": "#52514e",
                     "ytick.color": "#52514e", "font.size": 10, "axes.titleweight": "bold"})

def guardar(fig, nombre):
    fig.savefig(os.path.join(FIGURAS, nombre), bbox_inches="tight", dpi=150)

def met(m, c):
    return R["modelos"][m][c]["metricas"]

def auc(m, c):
    return met(m, c)["PROMEDIO"]["auc"]

def media(m, c, clave):
    return float(np.mean([v[clave] for k, v in met(m, c).items() if k != "PROMEDIO"]))

print("Imágenes evaluadas:", R["n_imagenes"])
print(f"Modelo nuevo: {CFG['n_train']:,} imágenes de entrenamiento, {CFG['n_val']:,} de validación interna, "
      f"mejor época {CFG['mejor_epoca']}")

## 1. ¿Cómo fue el entrenamiento? (paso 3)

AUC de **validación interna** por época (10 % de pacientes de CheXpert *train*, separado para elegir
la mejor época) y pérdida de entrenamiento. "Antes" es el modelo original sin entrenar.

⚠️ Este AUC **no es la prueba final**: esas imágenes tienen etiquetas automáticas y se usaron para
elegir la época. La comparación justa está en la sección 2 (CheXpert valid, etiquetado por radiólogos).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ep = [0] + list(HIST.epoca); a = [CFG["auc_val_inicial"]] + list(HIST.auc_val_7comunes)
ax[0].plot(ep, a, "-o", color=COLOR[NUEVO], lw=2, ms=7, zorder=3)
for x, y in zip(ep, a):
    ax[0].annotate(f"{y:.3f}", (x, y), textcoords="offset points", xytext=(0, 9), ha="center", fontsize=8, color="#52514e")
mejor = HIST.loc[HIST.auc_val_7comunes.idxmax()]
ax[0].plot(mejor.epoca, mejor.auc_val_7comunes, "o", ms=13, mfc="none", mec="#1baf7a", mew=2, zorder=4)
ax[0].annotate("mejor época (guardada)", (mejor.epoca, mejor.auc_val_7comunes),
               xytext=(max(mejor.epoca - 2.5, 0.5), min(a) + 0.04), fontsize=8, color="#1baf7a",
               arrowprops=dict(arrowstyle="->", color="#1baf7a"))
ax[0].set_xticks(ep, ["antes"] + [str(e) for e in HIST.epoca])
ax[0].set_ylim(min(a) - 0.03, max(a) + 0.03)
ax[0].set_xlabel("Época"); ax[0].set_ylabel("AUC promedio (7 patologías)")
ax[0].set_title(f"AUC de validación interna ({CFG['n_val']:,} imágenes)", fontsize=10)
ax[1].plot(HIST.epoca, HIST.perdida_train, "-o", color=COLOR[ORIG], lw=2, ms=7, zorder=3)
for x, y in zip(HIST.epoca, HIST.perdida_train):
    ax[1].annotate(f"{y:.3f}", (x, y), textcoords="offset points", xytext=(0, 9), ha="center", fontsize=8, color="#52514e")
ax[1].set_xticks(list(HIST.epoca))
ax[1].set_ylim(HIST.perdida_train.min() - 0.02, HIST.perdida_train.max() + 0.02)
ax[1].set_xlabel("Época"); ax[1].set_ylabel("Pérdida de entrenamiento (BCE)")
ax[1].set_title(f"Pérdida de entrenamiento ({CFG['n_train']:,} imágenes)", fontsize=10)
fig.suptitle("Ajuste fino del ResNet-50 a 320 px con CheXpert small", fontweight="bold")
fig.tight_layout(); guardar(fig, "9_entrenamiento_resnet320.png"); plt.show()

h = HIST.copy(); h["minutos"] = (h.segundos / 60).round(0)
display(h[["epoca", "perdida_train", "auc_val_7comunes", "minutos"]].round(4))
print(f"Tiempo total de entrenamiento: {HIST.segundos.sum() / 3600:.1f} horas")

## 2. Comparación principal: original vs nuevo (mismo protocolo, paso 2)

Mismas imágenes y mismas métricas para los dos modelos. **CheXpert valid** (202 imágenes frontales
etiquetadas por radiólogos) es el hospital de destino; **NIH 320 px** (25.596 imágenes) muestra si el
modelo nuevo olvidó lo que sabía del otro hospital.

Sensibilidad y especificidad son el promedio de las 7 patologías, cada modelo con **su propio umbral**
(el original, el de la librería; el nuevo, el de Youden calculado en su validación interna).

In [ ]:
filas = []
for c, nombre in CONDICIONES.items():
    for clave, etiqueta in [("auc", "AUC"), ("sensibilidad", "Sensibilidad"), ("especificidad", "Especificidad"), ("brier", "Brier")]:
        o = auc(ORIG, c) if clave == "auc" else media(ORIG, c, clave)
        n = auc(NUEVO, c) if clave == "auc" else media(NUEVO, c, clave)
        filas.append({"Condición": nombre, "Métrica": etiqueta, "Original": o, "Nuevo": n, "Diferencia": n - o})
comp = pd.DataFrame(filas).set_index(["Condición", "Métrica"])
display(comp.round(3))
print("Brier: más bajo es mejor (probabilidades mejor calibradas).")

## 3. ¿Vale la pena reentrenar? Todas las opciones en un gráfico

Para cada hospital, el AUC del ResNet-50 original a 320 px, las mejores técnicas **sin entrenar**
(paso 2b), el modelo **reentrenado** y, como referencia, el mejor modelo de la librería en CheXpert.

In [ ]:
def opciones(dom, c):
    o = [("ResNet-50 original (320 px)", auc(ORIG, c), COLOR[ORIG])]
    if T is not None:
        claves = ["320 interp. vecino", "320 AdaBN", "320 AdaBN + TTA"]
        for k in claves:
            if k in T[dom]:
                o.append(("+ " + k.replace("320 ", "").replace("interp.", "interpolación"),
                          T[dom][k]["metricas"]["PROMEDIO"]["auc"], COLOR["tec"]))
    o.append(("ResNet-50 nuevo (reentrenado)", auc(NUEVO, c), COLOR[NUEVO]))
    if REF in R["modelos"]:
        o.append(("DenseNet CheXpert (librería)", auc(REF, c), COLOR["ref"]))
    return o

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=False)
for ax, (dom, c, titulo) in zip(axes, [("chex", "chex_valid", "CheXpert valid (hospital de destino)"),
                                        ("nih", "nih_320", "NIH test a 320 px (hospital original)")]):
    o = opciones(dom, c)
    y = np.arange(len(o))
    ax.barh(y, [v for _, v, _ in o], color=[col for *_, col in o], height=0.6, zorder=3)
    for i, (_, v, _) in enumerate(o):
        ax.text(v + 0.004, i, f"{v:.3f}", va="center", fontsize=9)
    ax.set_yticks(y, [n for n, _, _ in o]); ax.invert_yaxis()
    ax.set_xlim(0.6, 0.88); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
    ax.axvline(auc(ORIG, c), color=COLOR[ORIG], lw=1, ls="--")
    ax.set_xlabel("AUC promedio (7 patologías)"); ax.set_title(titulo, fontsize=10)
fig.suptitle("Sin entrenar vs reentrenar (azul claro = técnicas sin entrenar, paso 2b)", fontweight="bold")
fig.tight_layout(); guardar(fig, "10_reentrenar_vs_sin_entrenar.png"); plt.show()

## 4. AUC por patología con intervalos de confianza del 95 %

Cada fila une el AUC del modelo original (azul) con el del nuevo (naranjo); las barras horizontales
son el IC95 % (bootstrap por paciente). Si los intervalos **no se tocan**, la diferencia es clara.
En CheXpert, neumonía (8 casos) y neumotórax (7 casos) tienen intervalos muy anchos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, c in zip(axes, ["chex_valid", "nih_320"]):
    y = np.arange(len(PATOLOGIAS))
    for i, p in enumerate(PATOLOGIAS):
        o, n = met(ORIG, c)[p], met(NUEVO, c)[p]
        ax.plot([o["auc"], n["auc"]], [i, i], color="#d6d5d0", lw=3, zorder=1)
        for d, col, dy in [(o, COLOR[ORIG], -0.12), (n, COLOR[NUEVO], 0.12)]:
            ax.errorbar(d["auc"], i + dy, xerr=[[d["auc"] - d["auc_ic95"][0]], [d["auc_ic95"][1] - d["auc"]]],
                        fmt="o", color=col, ms=7, capsize=3, lw=1.5, zorder=3)
    ax.set_yticks(y, [f"{ES[p]} ({met(NUEVO, c)[p]['n_pos']:,} casos)" for p in PATOLOGIAS])
    ax.invert_yaxis(); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
    ax.set_xlim(0.3, 1.0); ax.set_xlabel("AUC [IC95 %]")
    ax.set_title(CONDICIONES[c], fontsize=10)
axes[0].plot([], [], "o", color=COLOR[ORIG], label="Original"); axes[0].plot([], [], "o", color=COLOR[NUEVO], label="Nuevo")
axes[0].legend(frameon=False, loc="lower left")
fig.suptitle("AUC por patología: original vs nuevo", fontweight="bold")
fig.tight_layout(); guardar(fig, "11_por_patologia_original_vs_nuevo.png"); plt.show()

filas = []
for c in ["chex_valid", "nih_320"]:
    for p in PATOLOGIAS:
        o, n = met(ORIG, c)[p], met(NUEVO, c)[p]
        separados = o["auc_ic95"][1] < n["auc_ic95"][0] or n["auc_ic95"][1] < o["auc_ic95"][0]
        filas.append({"Condición": CONDICIONES[c], "Patología": ES[p], "Casos": n["n_pos"],
                      "AUC original": f"{o['auc']:.3f} [{o['auc_ic95'][0]:.2f}-{o['auc_ic95'][1]:.2f}]",
                      "AUC nuevo": f"{n['auc']:.3f} [{n['auc_ic95'][0]:.2f}-{n['auc_ic95'][1]:.2f}]",
                      "Diferencia": round(n["auc"] - o["auc"], 3),
                      "¿IC separados?": "sí" if separados else "no"})
display(pd.DataFrame(filas).set_index(["Condición", "Patología"]))

## 5. Sensibilidad y especificidad en CheXpert valid

El original casi nunca dice "enfermo" en CheXpert (sensibilidad cercana a 0). El nuevo, con sus
umbrales de Youden, detecta la mayoría de los casos a cambio de más falsas alarmas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
x = np.arange(len(PATOLOGIAS)); w = 0.38
for ax, clave in zip(axes, ["sensibilidad", "especificidad"]):
    for dx, m in [(-w / 2, ORIG), (w / 2, NUEVO)]:
        ax.bar(x + dx, [met(m, "chex_valid")[p][clave] for p in PATOLOGIAS], w - 0.04,
               color=COLOR[m], label=NOMBRE[m], zorder=3)
    ax.set_xticks(x, [ES[p] for p in PATOLOGIAS], rotation=30, ha="right")
    ax.set_ylim(0, 1.05); ax.set_title(clave.capitalize() + " (CheXpert valid)", fontsize=10)
axes[0].legend(frameon=False, loc="upper left")
fig.tight_layout(); guardar(fig, "12_sens_espec_chexpert.png"); plt.show()

## 6. Veredicto de la hipótesis H1

> **H1:** un ResNet-50 nuevo a 320 px, ajustado con CheXpert small, mejora el AUC en CheXpert valid
> frente al modelo original **sin empeorar** en NIH a 320 px.

Se calcula automáticamente. Como margen de "no empeorar" se usa 0,01 de AUC.
La prueba formal (bootstrap pareado) queda como siguiente paso.

In [ ]:
MARGEN = 0.01
d_chex = auc(NUEVO, "chex_valid") - auc(ORIG, "chex_valid")
d_nih = auc(NUEVO, "nih_320") - auc(ORIG, "nih_320")
parte1, parte2 = d_chex > 0, d_nih > -MARGEN
print(f"Parte 1 - mejora en CheXpert valid: {auc(ORIG, 'chex_valid'):.3f} -> {auc(NUEVO, 'chex_valid'):.3f} "
      f"({d_chex:+.3f})  {'CUMPLE' if parte1 else 'NO CUMPLE'}")
print(f"Parte 2 - no empeora en NIH 320 px: {auc(ORIG, 'nih_320'):.3f} -> {auc(NUEVO, 'nih_320'):.3f} "
      f"({d_nih:+.3f})  {'CUMPLE' if parte2 else 'NO CUMPLE'}")
print()
if parte1 and parte2:
    print("H1 se cumple completa.")
elif parte1:
    print("H1 se cumple a medias: el modelo mejora en el hospital de destino pero pierde en el original")
    print("(olvido catastrófico: se especializó en CheXpert).")
else:
    print("H1 no se cumple.")
if REF in R["modelos"]:
    print(f"\nReferencia: el mejor modelo de la librería en CheXpert ({NOMBRE[REF]}) tiene {auc(REF, 'chex_valid'):.3f}.")

## 7. Grad-CAM en CheXpert valid: ¿dónde mira cada modelo? (usa la GPU)

Radiografías **reales** de CheXpert valid que los radiólogos marcaron como positivas para la patología
elegida. Se muestra la misma imagen (320 px) con el mapa Grad-CAM del modelo **original** y del
**nuevo**, y la probabilidad que da cada uno. Rojo = zonas que más empujan la predicción.

CheXpert no trae cajas de radiólogo; para Cardiomegalia, lo esperable es que el mapa caiga sobre la
silueta del corazón (centro-abajo, algo hacia la derecha de la imagen).

Ajusta `CARPETA_CHEX` si tu CheXpert está en otra ruta. Cambia `PATOLOGIA` o `SEMILLA` para ver otros casos.

In [ ]:
CARPETA_CHEX = r"C:\Users\mines\Downloads\torchxrayvision-1.5.7\Chexpert small"
CARPETA_NIH = r"C:\Users\mines\Downloads\torchxrayvision-1.5.7\NIH"
PATOLOGIA = "Cardiomegaly"        # también: Effusion, Edema, Atelectasis, Consolidation
N_EJEMPLOS = 3
SEMILLA = 0                       # cambia el número para ver otras imágenes

import torch, torch.nn.functional as F, cv2
sys.path.insert(0, AQUI)
import comun_proyecto as cp
import torchxrayvision as xrv

device = "cuda" if torch.cuda.is_available() else "cpu"
modelos_cam = {}
if os.path.isdir(CARPETA_CHEX) or os.path.isdir(os.path.join(CARPETA_NIH, "images")):
    modelos_cam[ORIG] = cp.cargar_modelo(ORIG, device)[:2]
    modelos_cam[NUEVO] = cp.cargar_modelo(CARPETA_MODELO, device)[:2]

def gradcam(nombre, x, tam):
    """Grad-CAM de la última capa convolucional (layer4). x: tensor 1x1xHxW ya normalizado."""
    modelo, targets = modelos_cam[nombre]
    ti = targets.index(PATOLOGIA)
    act = {}
    h = modelo.model.layer4.register_forward_hook(lambda m_, i_, o: act.__setitem__("a", o))
    try:
        with torch.enable_grad(), cp.bench.quiet():
            y = modelo(x.to(device).float())
            g = torch.autograd.grad(y[0, ti], act["a"])[0][0]
    finally:
        h.remove()
    cam = F.relu((g.mean((1, 2))[:, None, None] * act["a"][0]).sum(0)).detach()
    cam = F.interpolate(cam[None, None], size=tam, mode="bilinear")[0, 0].cpu().numpy()
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8), float(y[0, ti])

if not os.path.isfile(os.path.join(CARPETA_CHEX, "valid.csv")):
    print("No se encontró", os.path.join(CARPETA_CHEX, "valid.csv"), "- ajusta CARPETA_CHEX para ver esta sección.")
else:
    dchex = cp.dataset_chexpert(CARPETA_CHEX, "valid.csv", res=320)
    col = dchex.pathologies.index(PATOLOGIA)
    positivos = np.where(dchex.labels[:, col] == 1)[0]
    rng = np.random.default_rng(SEMILLA)
    elegidos = rng.choice(positivos, min(N_EJEMPLOS, len(positivos)), replace=False)
    print(f"{len(positivos)} imágenes positivas para {PATOLOGIA} en CheXpert valid; se muestran {len(elegidos)}.")
    fig, axes = plt.subplots(len(elegidos), 3, figsize=(10.5, 3.7 * len(elegidos)), squeeze=False)
    for fila, i in zip(axes, elegidos):
        img = dchex[i]["img"]                                  # 1x320x320, ya normalizada
        ruta = dchex.csv.Path.iloc[i].split("valid/")[-1]
        vista = (img[0] - img.min()) / (img.max() - img.min())
        fila[0].imshow(vista, cmap="gray"); fila[0].set_title(f"{ruta}\n(radiólogo: {ES[PATOLOGIA]} = sí)", fontsize=8)
        for ax, m in zip(fila[1:], [ORIG, NUEVO]):
            cam, p = gradcam(m, torch.from_numpy(img)[None], (320, 320))
            ax.imshow(vista, cmap="gray"); ax.imshow(cam, cmap="jet", alpha=0.4)
            ax.set_title(f"{NOMBRE[m]}\nprob. {ES[PATOLOGIA]}: {p:.2f}", fontsize=9, color=COLOR[m])
        for ax in fila:
            ax.axis("off")
    fig.suptitle(f"Grad-CAM en CheXpert valid ({ES[PATOLOGIA]})", fontweight="bold")
    fig.tight_layout(); guardar(fig, f"13_gradcam_chexpert_{PATOLOGIA}.png"); plt.show()

## 8. Grad-CAM en NIH test, con la caja del radiólogo

Radiografías reales de NIH test con la **caja verde** que dibujaron radiólogos (lista oficial de NIH).
Columnas: original con la imagen completa (512 px), original a 320 px y nuevo a 320 px.
Sirve para ver si el modelo nuevo sigue mirando la lesión en el hospital que no vio al entrenarse.

In [ ]:
if not os.path.isdir(os.path.join(CARPETA_NIH, "images")):
    print("No se encontró", os.path.join(CARPETA_NIH, "images"), "- ajusta CARPETA_NIH para ver esta sección.")
else:
    cajas = pd.read_csv(os.path.join(cp.RAIZ, "torchxrayvision", "data", "BBox_List_2017.csv.gz"))
    cajas.columns = ["img", "label", "x", "y", "w", "h"] + list(cajas.columns[6:])
    test = cp.leer_lista(os.path.join(CARPETA_NIH, "test_list.txt"))
    cajas = cajas[(cajas.label == PATOLOGIA) & cajas.img.isin(test)]
    cajas = cajas.sample(min(N_EJEMPLOS, len(cajas)), random_state=SEMILLA) if len(cajas) else cajas
    if len(cajas) == 0:
        print("No hay imágenes de NIH test con caja para", PATOLOGIA)
    else:
        def tensor(img1024, res):
            img = cv2.resize(img1024, (res, res), interpolation=cv2.INTER_AREA)
            return torch.from_numpy(xrv.utils.normalize(img.astype(np.float32), 255))[None, None]
        fig, axes = plt.subplots(len(cajas), 4, figsize=(14, 3.8 * len(cajas)), squeeze=False)
        for fila, (_, c) in zip(axes, cajas.iterrows()):
            img = cv2.imread(os.path.join(CARPETA_NIH, "images", c.img), cv2.IMREAD_GRAYSCALE)
            fila[0].imshow(img, cmap="gray"); fila[0].set_title(f"{c.img}\n(caja verde = radiólogo)", fontsize=8)
            for ax, (m, res, titulo) in zip(fila[1:], [(ORIG, 512, "Original, 512 px"),
                                                     (ORIG, 320, "Original, 320 px"),
                                                     (NUEVO, 320, "Nuevo, 320 px")]):
                cam, p = gradcam(m, tensor(img, res), img.shape)
                ax.imshow(img, cmap="gray"); ax.imshow(cam, cmap="jet", alpha=0.4)
                ax.set_title(f"{titulo}\nprob. {ES[PATOLOGIA]}: {p:.2f}", fontsize=9, color=COLOR[m])
            for ax in fila:
                ax.add_patch(plt.Rectangle((c.x, c.y), c.w, c.h, fill=False, ec="#1baf7a", lw=2))
                ax.axis("off")
        fig.suptitle(f"Grad-CAM en NIH test ({ES[PATOLOGIA]})", fontweight="bold")
        fig.tight_layout(); guardar(fig, f"14_gradcam_nih_{PATOLOGIA}.png"); plt.show()

## 9. Hallazgos automáticos (se recalculan con cada corrida)

In [ ]:
chex_o, chex_n = auc(ORIG, "chex_valid"), auc(NUEVO, "chex_valid")
nih_o, nih_n = auc(ORIG, "nih_320"), auc(NUEVO, "nih_320")
print(f"- En CheXpert valid el AUC sube de {chex_o:.3f} a {chex_n:.3f} ({chex_n - chex_o:+.3f}).")
if REF in R["modelos"]:
    r = auc(REF, "chex_valid")
    print(f"  {'Supera' if chex_n > r else 'No supera'} al mejor modelo de la librería en CheXpert ({r:.3f}).")
print(f"- En NIH 320 px el AUC pasa de {nih_o:.3f} a {nih_n:.3f} ({nih_n - nih_o:+.3f}).")
if T is not None and "320 AdaBN" in T["chex"]:
    ada = T["chex"]["320 AdaBN"]["metricas"]["PROMEDIO"]["auc"]
    frac = (ada - chex_o) / (chex_n - chex_o) if chex_n != chex_o else np.nan
    print(f"- AdaBN sin entrenar llega a {ada:.3f} en CheXpert: consigue el {100 * frac:.0f} % de la mejora del ajuste fino,")
    print("  sin etiquetas y sin horas de GPU.")
ganadas = [ES[p] for p in PATOLOGIAS if met(NUEVO, "chex_valid")[p]["auc_ic95"][0] > met(ORIG, "chex_valid")[p]["auc_ic95"][1]]
perdidas = [ES[p] for p in PATOLOGIAS if met(NUEVO, "nih_320")[p]["auc_ic95"][1] < met(ORIG, "nih_320")[p]["auc_ic95"][0]]
print(f"- Mejoras con IC95 % separados en CheXpert: {', '.join(ganadas) or 'ninguna'}.")
print(f"- Pérdidas con IC95 % separados en NIH 320 px: {', '.join(perdidas) or 'ninguna'}.")
print(f"- Sensibilidad media en CheXpert: {media(ORIG, 'chex_valid', 'sensibilidad'):.2f} -> "
      f"{media(NUEVO, 'chex_valid', 'sensibilidad'):.2f}; especificidad: "
      f"{media(ORIG, 'chex_valid', 'especificidad'):.2f} -> {media(NUEVO, 'chex_valid', 'especificidad'):.2f}.")
print(f"- Brier medio en NIH 320 px: {media(ORIG, 'nih_320', 'brier'):.3f} -> {media(NUEVO, 'nih_320', 'brier'):.3f} "
      "(peor calibración: el modelo aprendió la prevalencia más alta de CheXpert).")
print("- Recordar: el original se entrenó con NIH (juega en casa allí); parte de la caída en NIH es perder esa ventaja.")